In [ ]:
!pip install tensorboardX

import argparse
def args_parser():
    parser = argparse.ArgumentParser()
    # federated arguments
    parser.add_argument('--epochs', type=int, default=10, help="rounds of training")
    parser.add_argument('--num_users', type=int, default=100, help="number of users: K")
    parser.add_argument('--frac', type=float, default=0.1, help='the fraction of clients: C')
    parser.add_argument('--local_ep', type=int, default=5, help="the number of local epochs: E")
    parser.add_argument('--local_bs', type=int, default=10, help="local batch size: B")
    parser.add_argument('--lr', type=float, default=0.001, help='learning rate')
    parser.add_argument('--momentum', type=float, default=0.5, help='SGD momentum (default: 0.5)')

    # model arguments
    parser.add_argument('--model', type=str, default='mlp', help='model name')
    parser.add_argument('--kernel_num', type=int, default=9, help='number of each kind of kernel')
    parser.add_argument('--kernel_sizes', type=str, default='3,4,5',
                        help='comma-separated kernel size to use for convolution')
    parser.add_argument('--norm', type=str, default='batch_norm', help="batch_norm, layer_norm, or None")
    parser.add_argument('--num_filters', type=int, default=32,
                        help="number of filters for conv nets -- 32 for miniimagenet, 64 for omiglot.")
    parser.add_argument('--max_pool', type=str, default='True',
                        help="Whether use max pooling rather than strided convolutions")

    # other arguments
    parser.add_argument('--dataset', type=str, default='mnist', help="name of dataset")
    parser.add_argument('--iid', type=int, default=0, help='whether i.i.d or not, 1 for iid, 0 for non-iid')
    parser.add_argument('--num_classes', type=int, default=10, help="number of classes")
    parser.add_argument('--num_channels', type=int, default=3, help="number of channels of images")
    parser.add_argument('--gpu', type=int, default=-1, help="GPU ID")
    parser.add_argument('--stopping_rounds', type=int, default=10, help='rounds of early stopping')
    parser.add_argument('--verbose', type=int, default=1, help='verbose print, 1 for True, 0 for False')
    parser.add_argument('--seed', type=int, default=1, help='random seed (default: 1)')
    parser.add_argument('--ag_scalar', type=float, default=1.0, help="global aggregation updating scalar, simplicity for A Matrix")
    parser.add_argument('--lg_scalar', type=float, default=1.0, help="client local updating scalar, simplicity for S Matrix")
    parser.add_argument('--algorithm', type=str, default='fedavg', help='algorithm for optimization')

    args, _ = parser.parse_known_args()  # Extract only the args
    return args

args = args_parser()
print(f"Number of epochs: {args.epochs}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.5/87.5 kB 3.8 MB/s eta 0:00:00
Number of epochs: 10


In [ ]:
!git clone https://github.com/JJisbug/UnreliableClientsinFL

Cloning into 'UnreliableClientsinFL'...
remote: Enumerating objects: 15, done.
remote: Counting objects: 100% (15/15), done.
remote: Compressing objects: 100% (15/15), done.
remote: Total 15 (delta 1), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (15/15), 15.67 KiB | 3.13 MiB/s, done.
Resolving deltas: 100% (1/1), done.


In [ ]:
cd /content/UnreliableClientsinFL

/content/UnreliableClientsinFL


In [ ]:
# 1'li user atmalı

import matplotlib
import random

matplotlib.use('Agg')
import matplotlib.pyplot as plt
import os
import copy
import numpy as np
from torchvision import datasets, transforms
import torch
import torch.nn.functional as F
from torch import autograd
from tensorboardX import SummaryWriter
from scipy import stats
import math

from sampling import mnist_iid, mnist_noniid, cifar_iid, cifar_noniid, server_test
#from options import args_parser
from Update import LocalFSVGRUpdate, LocalUpdate, ServerUpdate, LocalFedProxUpdate
from FedNets import MLP1, CNNMnist, CNNCifar
from averaging import average_weights



def test(net_g, data_loader, args):
    test_loss = 0
    correct = 0

    for idx, (data, target) in enumerate(data_loader):
        if args.gpu != -1:
            data, target = data.cuda(), target.cuda()
        data, target = autograd.Variable(data), autograd.Variable(target)
        log_probs = net_g(data)
        test_loss += F.nll_loss(log_probs, target, size_average=False).data[0]  # sum up batch loss
        y_pred = log_probs.data.max(1, keepdim=True)[1]  # get the index of the max log-probability
        correct += y_pred.eq(target.data.view_as(y_pred)).long().cpu().sum()

    test_loss /= len(data_loader.dataset)
    print('\nTest set: Average loss: {:.4f} \nAccuracy: {}/{} ({:.2f}%)\n'.format(
        test_loss, correct, len(data_loader.dataset),
        100. * correct / len(data_loader.dataset)))
    return correct, test_loss


def calculate_avg_grad(users_g):
    avg_grad = np.zeros(users_g[0][1].shape)
    total_size = np.sum([u[0] for u in users_g])  # Total number of samples of all users
    for i in range(len(users_g)):
        avg_grad = np.add(avg_grad,
                          users_g[i][0] * users_g[i][1])  # += users_g[i][0] * users_g[i][1]   <=> n_k * grad_k
    avg_grad = np.divide(avg_grad, total_size)  # /= total_size
    # print("avg_grad:", avg_grad)
    return avg_grad


def traverseList(nestList):
    flatList = []
    for item in nestList:
        if isinstance(item, list):
            flatList.extend(traverseList(item))
        else:
            flatList.append(item)
    return flatList


def flat(w):
    a = []
    for this_key in w.keys():
        a.append(list(copy.deepcopy(w[this_key].flatten().cpu().numpy())))
    return traverseList(a)


np.set_printoptions(threshold=100000)

if __name__ == '__main__':

    args = args_parser()
    algo_list = ['fedavg', 'fedprox', 'fsvgr']
    path_project = os.path.abspath('..')
    summary = SummaryWriter('local')
    args.gpu = -1  # -1 (CPU only)
    args.lr = 0.002  # 0.001 for cifar dataset
    args.model = 'cnn'  # 'mlp' or 'cnn'
    args.dataset = 'mnist'  # 'cifar' or 'mnist'
    args.num_users = 8
    args.local_bs = 5
    args.probability = 1  # Probability of malicious user attack
    args.global_test = 2000


    args.malicious = True
    args.defense = False
    args.malicious_num = 1
    args.iter_num = 0
    args.q = 1
    args.iid = True
    args.verbose = False
    args.num_total_users = 8
    args.num_in_groups = 2
    args.malicious_num = 7

    print("dataset:", args.dataset, " num_users:", args.num_users, "local_bs:", args.local_bs)
    if args.malicious:
        print("malicious_num", args.malicious_num, "iter_num", args.iter_num, "probability", args.probability)


    def Function(local_ep, T):
        dict_users = {}
        dataset_train = []
        w_globals = []
        accuracy_list = []
        loss_list = []
        malicious = 0
        malicious_number = -1
        selected = 0
        s_selected = 1
        fine_tune = 0
        i = 1
        p = 0
        if args.dataset == 'mnist':
            dataset_train = datasets.FashionMNIST('../data/fashionmnist/', train=True, download=True,
                                                  transform=transforms.Compose([
                                                      transforms.ToTensor(),
                                                      transforms.Normalize((0.1307,), (0.3081,))
                                                  ]))
            dataset_test = datasets.FashionMNIST('../data/fashionmnist/', train=False, download=True,
                                                 transform=transforms.Compose([
                                                     transforms.ToTensor(),
                                                     transforms.Normalize((0.1307,), (0.3081,))
                                                 ]))
            # sample users
            if args.iid:
                dict_users = mnist_iid(dataset_train, args.num_users)
            else:
                dict_users = mnist_noniid(dataset_train, args.num_users)
        else:
            exit('Error: unrecognized dataset')

        img_size = dataset_train[0][0].shape
        net_glob = None

        if args.model == 'cnn' and args.dataset == 'mnist':
          if args.gpu != -1:
              torch.cuda.set_device(args.gpu)
              net_glob = CNNMnist(args=args).cuda()
              print(f"Using GPU: {torch.cuda.get_device_name(args.gpu)}")
          else:
              net_glob = CNNMnist(args=args)
        else:
            exit('Error: unrecognized model')
        print("Nerual Net:", net_glob)
        net_glob.train()  # Train() does not change the weight values
        w_glob = net_glob.state_dict()
        w_size = 0
        for k in w_glob.keys():
            size = w_glob[k].size()
            if (len(size) == 1):
                nelements = size[0]
            else:
                nelements = size[0] * size[1]
            w_size += nelements * 4
            # print("Size ", k, ": ",nelements*4)
        print("Weight Size:", w_size, " bytes")
        print("Weight & Grad Size:", w_size * 2, " bytes")
        print("Each user Training size:", 784 * 8 / 8 * args.local_bs, " bytes")
        print("Total Training size:", 784 * 8 / 8 * 60000, " bytes")

        dataset_c_test = server_test(dataset_test, args.global_test)
        net_start = copy.deepcopy(net_glob)

        epoch_acc = []
        epoch_loss = []
        server_acc = np.zeros(args.num_total_users + int(args.num_total_users/args.num_in_groups)*3, dtype='float')
        net_server = copy.deepcopy(net_start)
        net_glob = copy.deepcopy(net_start)
        dataset_users_test = server_test(dataset_test, 2000)
        global_test = copy.deepcopy(dataset_c_test)
        person = [[] for _ in range(args.num_users)]
        w_list = []
        iter = 0
        exclude_value = []
        cıksın_mı_kalsın_mı=0
        glob_index = 1

        glob_iter_num = int(math.ceil(T / local_ep)) + 0
        exit_condition = int(glob_iter_num + np.log2(args.num_total_users)*2)
        epochs = exit_condition + 20
        bulunan_eleman = 0
        com = 0
        y11 = []
        y12 = []
        w_all_locals, loss_all_locals, acc_all_locals = [], [], []
        groups_selected =[[1, args.num_total_users/2], [args.num_total_users/2 + 1, args.num_total_users]]

        while iter <= epochs : # 0, 1, 2, 3, 4 ...
            iter += 1
            print(f"\n\niterations = {iter}:\n")
            com += local_ep
            epoch_server_acc = []
            w_locals, loss_locals, acc_locals = [], [], []
            sel_1 = iter - glob_iter_num - 1 - 2
            if (iter - glob_iter_num - 1) % 2 == 0 and iter > glob_iter_num + 1 and iter < exit_condition:
              print("Genel 2 epoch'da 1 kontrol noktası")
              if loss_list[sel_1] > loss_list[sel_1 + 1]:
                malicious_list = 0
              elif loss_list[sel_1] <= loss_list[sel_1 + 1]:
                malicious_list = 1

              idx = sel_1 + malicious_list
              group_first_add = int(groups_selected[idx][0])
              group_second_add = int(groups_selected[idx][0] + groups_selected[idx][1] - 1) / 2
              group_third_add = group_second_add + 1
              group_fourth_add = groups_selected[idx][1]
              groups_selected.append([group_first_add, group_second_add])
              groups_selected.append([group_third_add, group_fourth_add])


            if iter > 0 and iter <=glob_iter_num:
              print("\nGenel Eğitim\n")
              start_condition1 = 1
              finish_condition1 = args.num_total_users
            elif (iter > glob_iter_num) and iter <= exit_condition:
              print("\n2 ye bölmeli eğitim\n")
              start_condition1 = int(groups_selected[iter -glob_iter_num -1][0])
              finish_condition1 = int(groups_selected[iter -glob_iter_num -1][1])
            elif iter == exit_condition + 1:
              print("\nAyıklama evresi\n")
              if loss_list[sel_1] > loss_list[sel_1 + 1]:
                exclude_value.append(groups_selected[-2][0])
              elif loss_list[sel_1] <= loss_list[sel_1 + 1]:
                exclude_value.append(groups_selected[-1][0])
              start_condition1 = 1
              finish_condition1 = args.num_total_users
              bulunan_eleman = 1
            else:
              print(f"\nAtıldıktan sonraki eğitim => {exclude_value}\n")
              start_condition1 = 1
              finish_condition1 = args.num_total_users
              cıksın_mı_kalsın_mı = 1

            for rounds in range(1, 1 + 1):
                for idx in range(start_condition1, finish_condition1 + 1):
                    if idx in exclude_value:
                      continue
                    client_selected = idx
                    print("Seçilen kullanıcı:", client_selected)
                    local = LocalUpdate(args=args, dataset=dataset_train, idxs=dict_users[client_selected - 1], tb=summary)
                    w, loss, acc = local.update_weights(net=copy.deepcopy(net_glob), local_ep=int(local_ep))
                    if client_selected == args.malicious_num:
                        for this_key in w.keys():
                            dev = np.random.normal(0, 0.03, w[this_key].size())
                            dev = torch.from_numpy(dev).float()
                            w[this_key] = w[this_key] * (-1.5) + dev

                    else:
                      for this_key in w.keys():
                            dev = np.random.normal(0, 0.03, w[this_key].size())
                            dev = torch.from_numpy(dev).float()
                            w[this_key] = w[this_key] + dev
                    w_locals.append(copy.deepcopy(w))
                    loss_locals.append(copy.deepcopy(loss))
                    acc_locals.append(copy.deepcopy(acc))
                w_all_locals.append(copy.deepcopy(w_locals))
                loss_all_locals.append(copy.deepcopy(loss_locals))
                acc_all_locals.append(copy.deepcopy(acc_locals))

            for idx in range(finish_condition1 - start_condition1 + 1 - len(exclude_value)*cıksın_mı_kalsın_mı):
                net_server.load_state_dict(w_all_locals[rounds - 1][idx])
                server = ServerUpdate(args=args, dataset=dataset_test, idxs=dataset_users_test, tb=summary)
                acc, loss = server.test(net=net_server)
                epoch_server_acc.append(acc)
            weight = copy.deepcopy(epoch_server_acc)

            for i in range(len(weight)):
                weight[i] = 1
            total = sum(weight)
            for i in range(len(weight)):
                weight[i] /= total  # same weight average

            for i in range(len(w_locals)):
                w_list.append(copy.deepcopy(flat(w_locals[i])))

            w_glob = average_weights(w_locals, weight)
            w_globals.append(copy.deepcopy(w_glob))
            net_glob.load_state_dict(w_glob)
            net_local = ServerUpdate(args=args, dataset=dataset_test, idxs=global_test, tb=summary)
            return_acc, return_loss = net_local.test(net=net_glob)
            print("\nEpoch: {}, Global test loss {}, Global test acc: {:.2f}%\n".format(iter, return_loss,
                                                                                          100 * return_acc))
            if iter > glob_iter_num and iter <= exit_condition:
              loss_list.append(return_loss)
              accuracy_list.append(return_acc)
              w_glob = w_globals[iter - glob_index]
              glob_index += 1
              net_glob.load_state_dict(w_glob)
              net_local = ServerUpdate(args=args, dataset=dataset_test, idxs=global_test, tb=summary)

            y11.insert(iter, return_loss)
            y12.insert(iter, return_acc)

            if iter <= 200:
                if iter % 1 == 0 or iter == 0:
                    epoch_acc.append(return_acc)
                    epoch_loss.append(return_loss)
            else:
                if iter % 1 == 0:
                    epoch_acc.append(return_acc)
                    epoch_loss.append(return_loss)
        a = epoch_acc[-1]
        b = epoch_loss[-1]
        return a, b, y11, y12, epoch_acc, epoch_loss


    acc, loss, y1, y2, a, b = Function(256, 256*3)

dataset: mnist  num_users: 8 local_bs: 5
malicious_num 7 iter_num 0 probability 1
NN: CNNMnist is created
Nerual Net: CNNMnist(
  (conv1): Conv2d(1, 10, kernel_size=(5, 5), stride=(1, 1))
  (conv2): Conv2d(10, 20, kernel_size=(5, 5), stride=(1, 1))
  (fc1): Linear(in_features=320, out_features=50, bias=True)
  (fc2): Linear(in_features=50, out_features=10, bias=True)
)
Weight Size: 67200  bytes
Weight & Grad Size: 134400  bytes
Each user Training size: 3920.0  bytes
Total Training size: 47040000.0  bytes


iterations = 1:


Genel Eğitim

Seçilen kullanıcı: 1
Seçilen kullanıcı: 2
Seçilen kullanıcı: 3
Seçilen kullanıcı: 4
Seçilen kullanıcı: 5
Seçilen kullanıcı: 6
Seçilen kullanıcı: 7
Seçilen kullanıcı: 8

Epoch: 1, Global test loss 0.6855934858322144, Global test acc: 77.45%



iterations = 2:


Genel Eğitim

Seçilen kullanıcı: 1
Seçilen kullanıcı: 2
Seçilen kullanıcı: 3
Seçilen kullanıcı: 4
Seçilen kullanıcı: 5
Seçilen kullanıcı: 6
Seçilen kullanıcı: 7
Seçilen kullanıcı: 8

Epoch: 2, Gl

In [ ]:
# 09.10.2025

import matplotlib
import random
import time
import matplotlib.pyplot as plt
import os
import copy
import numpy as np
from torchvision import datasets, transforms
import torch
import torch.nn.functional as F
from torch import autograd
from tensorboardX import SummaryWriter
from scipy import stats
import math

from sampling import mnist_iid, mnist_noniid, cifar_iid, cifar_noniid, server_test
#from options import args_parser
from Update import LocalFSVGRUpdate, LocalUpdate, ServerUpdate, LocalFedProxUpdate
from FedNets import MLP1, CNNMnist, CNNCifar
from averaging import average_weights



def test(net_g, data_loader, args):
    test_loss = 0
    correct = 0

    for idx, (data, target) in enumerate(data_loader):
        if args.gpu != -1:
            data, target = data.cuda(), target.cuda()
        data, target = autograd.Variable(data), autograd.Variable(target)
        log_probs = net_g(data)
        test_loss += F.nll_loss(log_probs, target, size_average=False).data[0]  # sum up batch loss
        y_pred = log_probs.data.max(1, keepdim=True)[1]  # get the index of the max log-probability
        correct += y_pred.eq(target.data.view_as(y_pred)).long().cpu().sum()

    test_loss /= len(data_loader.dataset)
    print('\nTest set: Average loss: {:.4f} \nAccuracy: {}/{} ({:.2f}%)\n'.format(
        test_loss, correct, len(data_loader.dataset),
        100. * correct / len(data_loader.dataset)))
    return correct, test_loss


def calculate_avg_grad(users_g):
    avg_grad = np.zeros(users_g[0][1].shape)
    total_size = np.sum([u[0] for u in users_g])  # Total number of samples of all users
    for i in range(len(users_g)):
        avg_grad = np.add(avg_grad,
                          users_g[i][0] * users_g[i][1])  # += users_g[i][0] * users_g[i][1]   <=> n_k * grad_k
    avg_grad = np.divide(avg_grad, total_size)  # /= total_size
    # print("avg_grad:", avg_grad)
    return avg_grad


def traverseList(nestList):
    flatList = []
    for item in nestList:
        if isinstance(item, list):
            flatList.extend(traverseList(item))
        else:
            flatList.append(item)
    return flatList


def flat(w):
    a = []
    for this_key in w.keys():
        a.append(list(copy.deepcopy(w[this_key].flatten().cpu().numpy())))
    return traverseList(a)


np.set_printoptions(threshold=100000)

if __name__ == '__main__':
    start_time = time.time()
    args = args_parser()
    algo_list = ['fedavg', 'fedprox', 'fsvgr']
    path_project = os.path.abspath('..')
    summary = SummaryWriter('local')
    args.gpu = -1  # -1 (CPU only)
    args.lr = 0.002  # 0.001 for cifar dataset
    args.model = 'cnn'  # 'mlp' or 'cnn'
    args.dataset = 'mnist'  # 'cifar' or 'mnist'
    args.num_users = 8
    args.local_bs = 5
    args.probability = 1  # Probability of malicious user attack
    args.global_test = 2000

    args.malicious = True
    args.defense = False
    # args.malicious_num = [7,16]
    args.iter_num = 0
    args.q = 1
    args.iid = True
    args.verbose = False
    args.num_total_users = 8
    args.num_in_groups = 2
    args.malicious_num = [5, 6]


    print("dataset:", args.dataset, " num_users:", args.num_total_users, "local_bs:", args.local_bs)
    if args.malicious_num:
        print("malicious_num", args.malicious_num[0],args.malicious_num[1] ,"iter_num", args.iter_num, "probability", args.probability)


    def Function(local_ep, T):
        first_time = True
        dict_users = {}
        dataset_train = []
        w_globals = []
        accuracy_list = []
        loss_list = []
        malicious = 0
        malicious_number = -1
        selected = 0
        s_selected = 1
        exclude_value = []
        glob_index = 2
        last_2_group = []
        exit_ = 0
        cıksın_mı_kalsın_mı = 0
        p = 0
        if args.dataset == 'mnist':
            dataset_train = datasets.FashionMNIST('../data/fashionmnist/', train=True, download=True,
                                                  transform=transforms.Compose([
                                                      transforms.ToTensor(),
                                                      transforms.Normalize((0.1307,), (0.3081,))
                                                  ]))
            dataset_test = datasets.FashionMNIST('../data/fashionmnist/', train=False, download=True,
                                                 transform=transforms.Compose([
                                                     transforms.ToTensor(),
                                                     transforms.Normalize((0.1307,), (0.3081,))
                                                 ]))
            # sample users
            if args.iid:
                dict_users = mnist_iid(dataset_train, args.num_total_users)
            else:
                dict_users = mnist_noniid(dataset_train, args.num_total_users)
        else:
            exit('Error: unrecognized dataset')

        img_size = dataset_train[0][0].shape
        net_glob = None

        if args.model == 'cnn' and args.dataset == 'mnist':
          if args.gpu != -1:
              torch.cuda.set_device(args.gpu)
              net_glob = CNNMnist(args=args).cuda()
              print(f"Using GPU: {torch.cuda.get_device_name(args.gpu)}")
          else:
              net_glob = CNNMnist(args=args)
        else:
            exit('Error: unrecognized model')
        print("Nerual Net:", net_glob)
        net_glob.train()  # Train() does not change the weight values
        w_glob = net_glob.state_dict()
        w_size = 0
        for k in w_glob.keys():
            size = w_glob[k].size()
            if (len(size) == 1):
                nelements = size[0]
            else:
                nelements = size[0] * size[1]
            w_size += nelements * 4

        dataset_c_test = server_test(dataset_test, args.global_test)
        net_start = copy.deepcopy(net_glob)

        epoch_acc = []
        epoch_loss = []
        server_acc = np.zeros(args.num_total_users + int(args.num_total_users/args.num_in_groups)*3, dtype='float')
        net_server = copy.deepcopy(net_start)
        net_glob = copy.deepcopy(net_start)
        dataset_users_test = server_test(dataset_test, 2000)
        global_test = copy.deepcopy(dataset_c_test)

        person = [[] for _ in range(args.num_total_users)]
        w_list = []
        return_loss = 1
        iter = 0

        glob_iter_num = int(math.ceil(T / local_ep)) + 0
        # Bu değerin çift sayı olması gerek
        # Hiç bir eleman atılmadan bu değer kadar iterasyon başlangıçta yapılır.

        exit_condition = int(glob_iter_num + np.log2(args.num_total_users) + 9)
        epochs = exit_condition + 30
        bulunan_eleman = 0
        count_for_different_groups = True
        ilk_tur = True
        y11 = []
        y12 = []
        tek_mi_cift_mi = 0
        w_all_locals, loss_all_locals, acc_all_locals = [], [], []
        groups_selected =[[1, args.num_total_users/2], [args.num_total_users/2 + 1, args.num_total_users]]


        # for iter in range(1, epochs + 1): # 0, 1, 2, 3, 4 ...
        while iter <= epochs : # Seçilen epoch sayısına göre loss değerini ayarlamak gerekmekte.
            iter += 1
            epoch_server_acc = []
            w_locals, loss_locals, acc_locals = [], [], []

            print(f"Loss list = {loss_list}, iter = {iter}")

            if (iter) % 2 == 0 and (iter > glob_iter_num + 1) and (bulunan_eleman != 2):
              print("Genel 2 epoch'da 1 kontrol noktası")

              sel_1 = iter - glob_iter_num -3
              print(f"{sel_1}, {sel_1 + 1}")
              loss_farki = loss_list[sel_1] - loss_list[sel_1 + 1]

              if tek_mi_cift_mi != 2:
                if abs(loss_farki) < 0.035 :
                  #Hata listeleri arasındaki fark az, demek ki iki grupta birbirine çok benziyor. İki dalda da 1 tane malicious var.
                  tek_mi_cift_mi = 2
                elif abs(loss_farki) >= 0.035 :
                  #Hata listeleri arasındaki fark fazla, demek ki iki grupta birbirine benzemiyor. Tek dalda iki malicious var..
                  tek_mi_cift_mi = 1
              print(f"İterasyon = {iter}, maliciousların olduğu grup sayısı => {tek_mi_cift_mi}")
              if tek_mi_cift_mi == 1:
                  malicious = sel_1 + 1 if loss_farki < 0 else sel_1
                  malicious_range = [malicious]
                  print(f"Tek grup için adamlar {malicious_range} grubunda")

              elif tek_mi_cift_mi == 2:

                if ilk_tur == True:
                  malicious_range = [iter - glob_iter_num - 2 - 1, iter - glob_iter_num - 2]
                  # İlk turu fark ettiğimiz iter numarası önemli.Çünkü ona göre malicious gruplar seçilecek.
                  # -glob_iter_num çünkü başta o kadar boş tur dönüyor,
                  # -2 çünkü o ana kadar ki tek grubu 2 iter de parça parça kontrol ediyor,- 1 çünkü parça parça kontolden sonra işleme başlıyor.
                  ilk_tur = False

                elif first_time == True:
                  count_for_different_groups = True
                  malicious = sel_1 + 1 if loss_farki < 0 else sel_1
                  malicious_range = [malicious]
                  print(f"Çift grup için adamlar1 {malicious_range} grubunda")
                  first_time = False

                else:
                  malicious = sel_1 + 1 if loss_farki < 0 else sel_1
                  malicious_range = [malicious]
                  print(f"Çift grup için adamlar2 {malicious_range} grubunda")
                  first_time = True
                  count_for_different_groups = False

              # Fonksiyon haline getir. Tek dal için 2 iter de 1, Çift dal için 4 iter de 1.

              for malicious in malicious_range:

                  idx = malicious
                  group_first_add = int(groups_selected[idx][0])
                  group_second_add = int(groups_selected[idx][0] + groups_selected[idx][1] - 1) / 2
                  group_third_add = group_second_add + 1
                  group_fourth_add = groups_selected[idx][1]

                  print(f"\nHesaplamalar.\n{group_first_add}--{group_second_add}--{group_third_add}--{group_fourth_add}")

                  if (group_second_add != int(group_second_add)):
                    print("İlk malicious grubu bulundu baba!\n")
                    groups_selected.append([group_first_add, group_first_add])
                    groups_selected.append([group_first_add + 1, group_first_add + 1])
                    bulunan_eleman += 1
                    if (bulunan_eleman == 1 or bulunan_eleman == 2):
                      exclude_value.append(group_first_add)
                      print(f"Exculude values = {exclude_value}")
                  elif (group_first_add == group_second_add) and (group_third_add == group_fourth_add) and (tek_mi_cift_mi == 2):
                    print("İki ayrı dalda iki malicious bulundu")
                    groups_selected.append([group_first_add, group_second_add])
                    groups_selected.append([group_third_add, group_fourth_add])
                  elif (group_first_add == group_second_add) and (tek_mi_cift_mi == 1):
                    print("Tek dalda iki malicious bulundu")
                    bulunan_eleman = 2
                    exclude_value = groups_selected[malicious]#groups_selected[-1] if malicious == 2 else groups_selected[-2]
                  else:
                    print("NET bulunmadı devam")
                    groups_selected.append([group_first_add, group_second_add])
                    groups_selected.append([group_third_add, group_fourth_add])


            print(f"Gruplar = {groups_selected}")
            if iter > 0 and iter <=glob_iter_num:
              print("\nGenel Eğitim\n")
              start_condition1 = 1
              finish_condition1 = args.num_total_users
            elif ((iter > glob_iter_num) and (bulunan_eleman != 2)):
              print("\n2 ye bölmeli eğitim\n")
              start_condition1 = int(groups_selected[iter -glob_iter_num -1][0])
              finish_condition1 = int(groups_selected[iter -glob_iter_num -1][1])
            else:
              print(f"\nAtıldıktan sonraki eğitim => {exclude_value}\n")
              start_condition1 = 1
              finish_condition1 = args.num_total_users
              cıksın_mı_kalsın_mı = 1
            print(f"Başlangıç değeri = {start_condition1}, Bitiş değeri = {finish_condition1}")
            for rounds in range(1, 1 + 1):
                for idx in range(start_condition1, finish_condition1 + 1):
                    if idx in exclude_value:
                      continue
                    client_selected = idx
                    print("Seçilen kullanıcı:", client_selected)
                    local = LocalUpdate(args=args, dataset=dataset_train, idxs=dict_users[client_selected - 1], tb=summary)
                    w, loss, acc = local.update_weights(net=copy.deepcopy(net_glob), local_ep=int(local_ep))
                    for list_elements in args.malicious_num:
                      if client_selected == list_elements:
                          for this_key in w.keys():
                              dev = np.random.normal(0, 0.2, w[this_key].size())
                              dev = torch.from_numpy(dev).float()
                              w[this_key] = w[this_key] * (-1.5) #+ dev arttır.
                    w_locals.append(copy.deepcopy(w))
                    loss_locals.append(copy.deepcopy(loss))
                    acc_locals.append(copy.deepcopy(acc))
                w_all_locals.append(copy.deepcopy(w_locals))
                loss_all_locals.append(copy.deepcopy(loss_locals))
                acc_all_locals.append(copy.deepcopy(acc_locals))

            for idx in range(finish_condition1 - start_condition1 + 1 - len(exclude_value)*cıksın_mı_kalsın_mı):
                net_server.load_state_dict(w_all_locals[iter - 1][idx])
                server = ServerUpdate(args=args, dataset=dataset_test, idxs=dataset_users_test, tb=summary)
                acc, loss = server.test(net=net_server)
                # server_acc[server_acc_num + idx] += acc
                epoch_server_acc.append(acc)
            weight = copy.deepcopy(epoch_server_acc)

            for i in range(len(weight)):
                weight[i] = 1
            total = sum(weight)
            for i in range(len(weight)):
                weight[i] /= total  # same weight average
            for i in range(len(w_locals)):
                w_list.append(copy.deepcopy(flat(w_locals[i])))
            print(f"weight = {len(weight)}, locals = {len(w_locals)}")
            w_glob = average_weights(w_locals, weight)
            w_globals.append(copy.deepcopy(w_glob))
            net_glob.load_state_dict(w_glob)
            net_local = ServerUpdate(args=args, dataset=dataset_test, idxs=global_test, tb=summary)
            return_acc, return_loss = net_local.test(net=net_glob)
            print("\nEpoch: {}, Global test loss {}, Global test acc: {:.2f}%\n\n\n\n".format(iter, return_loss, 100 * return_acc))

            if iter > glob_iter_num and bulunan_eleman != 2:
              loss_list.append(return_loss)
              accuracy_list.append(return_acc)
              w_glob = w_globals[iter - glob_index]
              glob_index += 1
              net_glob.load_state_dict(w_glob)
              net_local = ServerUpdate(args=args, dataset=dataset_test, idxs=global_test, tb=summary)

            y11.insert(iter, return_loss)
            y12.insert(iter, return_acc)

            if iter <= 200:
                if iter % 1 == 0 or iter == 0:
                    epoch_acc.append(return_acc)
                    epoch_loss.append(return_loss)
            else:
                if iter % 1 == 0:
                    epoch_acc.append(return_acc)
                    epoch_loss.append(return_loss)
        a = epoch_acc[-1]
        b = epoch_loss[-1]
        return a, b, y11, y12, epoch_acc, epoch_loss
    acc, loss, y1, y2, a, b = Function(256, 256*3)
    end_time = time.time()
    elapsed_time = end_time - start_time
    minutes = int(elapsed_time // 60)  # Get whole minutes
    seconds = int(elapsed_time % 60)   # Get remaining seconds

ModuleNotFoundError: No module named 'tensorboardX'

In [ ]:
LST = [acc, loss, y1, y2, a, b]
for element in LST:
  print(element)

In [ ]:
################################################################################
import matplotlib
import random
import time
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import os
import copy
import numpy as np
from torchvision import datasets, transforms
import torch
import torch.nn.functional as F
from torch import autograd
from tensorboardX import SummaryWriter
from scipy import stats
import math

from sampling import mnist_iid, mnist_noniid, cifar_iid, cifar_noniid, server_test
#from options import args_parser
from Update import LocalFSVGRUpdate, LocalUpdate, ServerUpdate, LocalFedProxUpdate
from FedNets import MLP1, CNNMnist, CNNCifar
from averaging import average_weights



def test(net_g, data_loader, args):
    test_loss = 0
    correct = 0

    for idx, (data, target) in enumerate(data_loader):
        if args.gpu != -1:
            data, target = data.cuda(), target.cuda()
        data, target = autograd.Variable(data), autograd.Variable(target)
        log_probs = net_g(data)
        test_loss += F.nll_loss(log_probs, target, size_average=False).data[0]  # sum up batch loss
        y_pred = log_probs.data.max(1, keepdim=True)[1]  # get the index of the max log-probability
        correct += y_pred.eq(target.data.view_as(y_pred)).long().cpu().sum()

    test_loss /= len(data_loader.dataset)
    print('\nTest set: Average loss: {:.4f} \nAccuracy: {}/{} ({:.2f}%)\n'.format(
        test_loss, correct, len(data_loader.dataset),
        100. * correct / len(data_loader.dataset)))
    return correct, test_loss


def calculate_avg_grad(users_g):
    avg_grad = np.zeros(users_g[0][1].shape)
    total_size = np.sum([u[0] for u in users_g])  # Total number of samples of all users
    for i in range(len(users_g)):
        avg_grad = np.add(avg_grad,
                          users_g[i][0] * users_g[i][1])  # += users_g[i][0] * users_g[i][1]   <=> n_k * grad_k
    avg_grad = np.divide(avg_grad, total_size)  # /= total_size
    # print("avg_grad:", avg_grad)
    return avg_grad


def traverseList(nestList):
    flatList = []
    for item in nestList:
        if isinstance(item, list):
            flatList.extend(traverseList(item))
        else:
            flatList.append(item)
    return flatList


def flat(w):
    a = []
    for this_key in w.keys():
        a.append(list(copy.deepcopy(w[this_key].flatten().cpu().numpy())))
    return traverseList(a)


np.set_printoptions(threshold=100000)

if __name__ == '__main__':
    start_time = time.time()
    args = args_parser()
    algo_list = ['fedavg', 'fedprox', 'fsvgr']
    path_project = os.path.abspath('..')
    summary = SummaryWriter('local')
    args.gpu = 0  # -1 (CPU only)
    args.lr = 0.002  # 0.001 for cifar dataset
    args.model = 'cnn'  # 'mlp' or 'cnn'
    args.dataset = 'mnist'  # 'cifar' or 'mnist'
    args.num_users = 16
    args.local_bs = 5
    args.probability = 1  # Probability of malicious user attack
    args.global_test = 2000

    args.malicious = True
    args.defense = False
    args.malicious_num = 1
    args.iter_num = 0
    args.q = 1
    args.iid = True
    args.verbose = False
    args.num_total_users = 16
    args.num_in_groups = 2
    args.malicious_num = 7

    print("dataset:", args.dataset, " num_users:", args.num_total_users, "local_bs:", args.local_bs)
    if args.malicious:
        print("malicious_num", args.malicious_num, "iter_num", args.iter_num, "probability", args.probability)


    def Function(local_ep, T):
        dict_users = {}
        dataset_train = []
        w_globals = []
        accuracy_list = []
        malicious = 0
        malicious_number = -1
        selected = 0
        s_selected = 1
        fine_tune = 0
        exclude_value = 0
        exclude_i = 0
        glob_index = 2
        p = 0
        if args.dataset == 'mnist':
            dataset_train = datasets.FashionMNIST('../data/fashionmnist/', train=True, download=True,
                                                  transform=transforms.Compose([
                                                      transforms.ToTensor(),
                                                      transforms.Normalize((0.1307,), (0.3081,))
                                                  ]))
            dataset_test = datasets.FashionMNIST('../data/fashionmnist/', train=False, download=True,
                                                 transform=transforms.Compose([
                                                     transforms.ToTensor(),
                                                     transforms.Normalize((0.1307,), (0.3081,))
                                                 ]))
            # sample users
            if args.iid:
                dict_users = mnist_iid(dataset_train, args.num_total_users)
            else:
                dict_users = mnist_noniid(dataset_train, args.num_total_users)
        else:
            exit('Error: unrecognized dataset')

        img_size = dataset_train[0][0].shape
        net_glob = None

        if args.model == 'cnn' and args.dataset == 'mnist':
          if args.gpu != -1:
              torch.cuda.set_device(args.gpu)
              net_glob = CNNMnist(args=args).cuda()
              print(f"Using GPU: {torch.cuda.get_device_name(args.gpu)}")
          else:
              net_glob = CNNMnist(args=args)
        else:
            exit('Error: unrecognized model')
        print("Nerual Net:", net_glob)
        net_glob.train()  # Train() does not change the weight values
        w_glob = net_glob.state_dict()
        w_size = 0
        for k in w_glob.keys():
            size = w_glob[k].size()
            if (len(size) == 1):
                nelements = size[0]
            else:
                nelements = size[0] * size[1]
            w_size += nelements * 4

        dataset_c_test = server_test(dataset_test, args.global_test)
        net_start = copy.deepcopy(net_glob)

        epoch_acc = []
        epoch_loss = []
        server_acc = np.zeros(args.num_total_users + int(args.num_total_users/args.num_in_groups)*3, dtype='float')
        net_server = copy.deepcopy(net_start)
        net_glob = copy.deepcopy(net_start)
        dataset_users_test = server_test(dataset_test, 2000)
        global_test = copy.deepcopy(dataset_c_test)
        person = [[] for _ in range(args.num_total_users)]
        w_list = []
        return_loss = 1
        iter = 0
        co = 1 if np.log2(args.num_total_users) %2 == 0 else 0
        glob_iter_num = int(math.ceil(T / local_ep)) + co + 0#çift sayı olması lazım
        exit_condition = int(glob_iter_num + np.log2(args.num_total_users) + 2)


        epochs = exit_condition + 2 + 6
        com = 0
        y11 = []
        y12 = []
        w_all_locals, loss_all_locals, acc_all_locals = [], [], []
        groups_selected =[[1, args.num_total_users/2], [args.num_total_users/2 + 1, args.num_total_users]]
        # for iter in range(1, epochs + 1): # 0, 1, 2, 3, 4 ...
        while iter <= epochs or return_loss <= 0.525:
            iter += 1
            com += local_ep
            epoch_server_acc = []
            w_locals, loss_locals, acc_locals = [], [], []

            if ((iter + co) % 2 == 0 and ((iter > glob_iter_num + 1) and (iter <= exit_condition))): #glob_iter_num tek sayı olmalı
              print(f"Groups selected = {groups_selected}")
              malicious = 2 if accuracy_list[iter - glob_iter_num -3] > accuracy_list[iter - glob_iter_num -2] else 1
              groups_selected.append([groups_selected[iter -glob_iter_num -4 + malicious][0], (groups_selected[iter -glob_iter_num -4 +malicious][0] + groups_selected[iter -glob_iter_num -4 + malicious][1] - 1)/2])
              groups_selected.append([(groups_selected[iter -glob_iter_num -4 + malicious][0] + groups_selected[iter -glob_iter_num -4 + malicious][1] - 1)/2 + 1, groups_selected[iter -glob_iter_num -4 + malicious][1]])
            elif iter == exit_condition + 3:
              malicious = groups_selected[-1][1] if accuracy_list[-2] > accuracy_list[-1] else groups_selected[-1][0]

            if iter > 0 and iter <=glob_iter_num:
              print("\nGenel Eğitim\n")
              start_condition1 = 1
              finish_condition1 = args.num_total_users
            elif ((iter > glob_iter_num) and (iter <= exit_condition)):
              print("\n2 ye bölmeli eğitim\n")
              start_condition1 = int(groups_selected[iter -glob_iter_num -1][0])
              finish_condition1 = int(groups_selected[iter -glob_iter_num -1][1])
            elif iter == exit_condition + 1:
              print("\nİlk malicious eğitmi\n")
              start_condition1 = int(groups_selected[-1][0])
              finish_condition1 = int(groups_selected[-1][1])
            elif iter == exit_condition + 2:
              print("\nİkinci malicious eğitmi\n")
              start_condition1 = int(groups_selected[-1][0] + 1)
              finish_condition1 = int(groups_selected[-1][1] + 1)
            elif iter > exit_condition + 2:
              print(f"\nMalicious({malicious})atıldıktan sonraki eğitim\n")
              start_condition1 = 1
              finish_condition1 = args.num_total_users
              exclude_value = malicious # groups_selected[-1][1] if accuracy_list[-2] > accuracy_list[-1] else groups_selected[-1][0]
              exclude_i = -1
            print(f"start_condition1 = {start_condition1}, finish_condition1 = {finish_condition1}")
            for rounds in range(1, 1 + 1):
                for idx in range(start_condition1, finish_condition1 + 1):
                    if idx == exclude_value:
                      continue
                    client_selected = idx
                    print("client_selected:", client_selected)
                    local = LocalUpdate(args=args, dataset=dataset_train, idxs=dict_users[client_selected - 1], tb=summary)
                    w, loss, acc = local.update_weights(net=copy.deepcopy(net_glob), local_ep=int(local_ep))
                    if client_selected == args.malicious_num:
                        for this_key in w.keys():
                            dev = np.random.normal(0, 0.2, w[this_key].size())
                            dev = torch.from_numpy(dev).float()
                            w[this_key] = w[this_key] * (-0.9) #+ dev arttır.
                    w_locals.append(copy.deepcopy(w))
                    loss_locals.append(copy.deepcopy(loss))
                    acc_locals.append(copy.deepcopy(acc))
                w_all_locals.append(copy.deepcopy(w_locals))
                loss_all_locals.append(copy.deepcopy(loss_locals))
                acc_all_locals.append(copy.deepcopy(acc_locals))

            for idx in range(finish_condition1 - start_condition1 + 1 + exclude_i):
                net_server.load_state_dict(w_all_locals[iter - 1][idx])
                server = ServerUpdate(args=args, dataset=dataset_test, idxs=dataset_users_test, tb=summary)
                acc, loss = server.test(net=net_server)
                # server_acc[server_acc_num + idx] += acc
                epoch_server_acc.append(acc)
            weight = copy.deepcopy(epoch_server_acc)

            for i in range(len(weight)):
                weight[i] = 1
            total = sum(weight)
            for i in range(len(weight)):
                weight[i] /= total  # same weight average
            for i in range(len(w_locals)):
                w_list.append(copy.deepcopy(flat(w_locals[i])))
            print(f"weight = {len(weight)}, locals = {len(w_locals)}")
            w_glob = average_weights(w_locals, weight)
            w_globals.append(copy.deepcopy(w_glob))
            net_glob.load_state_dict(w_glob)
            net_local = ServerUpdate(args=args, dataset=dataset_test, idxs=global_test, tb=summary)
            return_acc, return_loss = net_local.test(net=net_glob)
            print("\nEpoch: {}, Global test loss {}, Global test acc: {:.2f}%\n\n\n\n".format(iter, return_loss, 100 * return_acc))
            if iter > glob_iter_num:
              accuracy_list.append(return_acc)
              w_glob = w_globals[iter - glob_index]
              glob_index += 1
              net_glob.load_state_dict(w_glob)
              net_local = ServerUpdate(args=args, dataset=dataset_test, idxs=global_test, tb=summary)
            y11.insert(iter, return_loss)
            y12.insert(iter, return_acc)

            if iter <= 200:
                if iter % 1 == 0 or iter == 0:
                    epoch_acc.append(return_acc)
                    epoch_loss.append(return_loss)
            else:
                if iter % 1 == 0:
                    epoch_acc.append(return_acc)
                    epoch_loss.append(return_loss)
        a = epoch_acc[-1]
        b = epoch_loss[-1]
        return a, b, y11, y12, epoch_acc, epoch_loss
    acc, loss, y1, y2, a, b = Function(16, 16*3)
    end_time = time.time()
    elapsed_time = end_time - start_time
    minutes = int(elapsed_time // 60)  # Get whole minutes
    seconds = int(elapsed_time % 60)   # Get remaining seconds
    print(f"Elapsed time: {minutes} minutes and {seconds} seconds")